# 23 — Spectral Analysis

Compare the frequency content of a healthy bearing against a degraded one using a raw FFT and Welch PSD.

**Dataset**: first study in any ISA-PHM dataset (assay 1, fs = 25 600 Hz)  
**API**: `assay.plot_frequency_domain()` · `assay.plot_psd()`
Run the import and dataset-loading cells first. Each following example selects what it needs independently; change its study, assay, or run to explore your data.


## Imports and dataset path


In [ ]:
from pathlib import Path

from isa_phm import ISAWrapper
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show

output_notebook()

# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## Load the wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
    cache_maxsize=10,
)


## Dataset overview


In [ ]:
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)   # horizontal accelerometer

all_runs  = assay.list_runs()
first_run = all_runs[0]
last_run  = all_runs[-1]

FS = 25_600.0   # Hz ? declared in ISA-JSON protocol parameters

print(f"Assay     : {assay.assay_id}  ({assay.run_count} runs)")
print(f"First run : {first_run.run_id}  (#{first_run.run_number})")
print(f"Last run  : {last_run.run_id}  (#{last_run.run_number})")
print(f"Fs        : {FS:,.0f} Hz")


## 1. Load both signals

Load the raw DataFrames so we can pass them directly to both plot methods.

In [ ]:
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)
first_run = assay.run(1)
last_run = assay.list_runs()[-1]

df_early = assay.load_dataframe(run_id=first_run.run_id, file_type="raw")
df_late  = assay.load_dataframe(run_id=last_run.run_id,  file_type="raw")

print(f"Early signal: {len(df_early):,} samples")
print(f"Late signal : {len(df_late):,} samples")

## 2. Raw FFT — healthy vs degraded

The raw FFT (single periodogram) shows the amplitude spectrum for each run.  
Fault frequencies appear as peaks that grow with severity.

In [ ]:
FS = 25_600.0  # Hz; change to match your dataset
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)
first_run = assay.run(1)
df_early = assay.load_dataframe(run_id=first_run.run_id, file_type="raw")

fig = assay.plot_frequency_domain(
    df=df_early,
    fs=FS,
    log_scale=True,
    title=f"FFT — Healthy (run {first_run.run_number})",
)
bokeh_show(fig)

In [ ]:
FS = 25_600.0  # Hz; change to match your dataset
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)
last_run = assay.list_runs()[-1]
df_late = assay.load_dataframe(run_id=last_run.run_id, file_type="raw")

fig = assay.plot_frequency_domain(
    df=df_late,
    fs=FS,
    log_scale=True,
    title=f"FFT — Near Failure (run {last_run.run_number})",
)
bokeh_show(fig)

## 3. Welch PSD — smoother spectral estimate

Welch's method averages overlapping periodograms, reducing spectral variance.  
This makes fault frequencies easier to identify in noisy real-world signals.

> `nperseg=1024` — segment length; larger = finer frequency resolution but less smoothing.

In [ ]:
FS = 25_600.0  # Hz; change to match your dataset
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)
first_run = assay.run(1)

fig = assay.plot_psd(
    run_id=first_run.run_id,
    fs=FS,
    nperseg=1024,
    file_type="raw",
    title=f"Welch PSD — Healthy (run {first_run.run_number})",
)
bokeh_show(fig)

In [ ]:
FS = 25_600.0  # Hz; change to match your dataset
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)
last_run = assay.list_runs()[-1]

fig = assay.plot_psd(
    run_id=last_run.run_id,
    fs=FS,
    nperseg=1024,
    file_type="raw",
    title=f"Welch PSD — Near Failure (run {last_run.run_number})",
)
bokeh_show(fig)

## 4. Effect of segment length on Welch PSD

A larger `nperseg` gives finer frequency resolution but less smoothing.  
Compare 256, 1024, and 4096 on the degraded signal.

In [ ]:
FS = 25_600.0  # Hz; change to match your dataset
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)
last_run = assay.list_runs()[-1]

for nperseg in [256, 1024, 4096]:
    fig = assay.plot_psd(
        run_id=last_run.run_id,
        fs=FS,
        nperseg=nperseg,
        file_type="raw",
        title=f"Welch PSD — Near Failure  (nperseg={nperseg})",
    )
    bokeh_show(fig)

## 5. Time-frequency spectrogram

Spectrogram shows how frequency content evolves within a single run.


In [ ]:
FS = 25_600.0  # Hz; change to match your dataset
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)
last_run = assay.list_runs()[-1]

fig = assay.plot_spectrogram(
    run_id=last_run.run_id,
    fs=FS,
    nperseg=512,
    overlap=0.75,
    file_type="raw",
    title=f"Spectrogram ? run {last_run.run_number}",
)
bokeh_show(fig)


## 6. Waterfall spectrum across lifecycle

Waterfall stacks spectra from multiple runs to show trend evolution.


In [ ]:
FS = 25_600.0  # Hz; change to match your dataset
study = wrapper.study("Bearing 1_1")
assay = study.assay(1)

fig = assay.plot_waterfall(
    fs=FS,
    n_runs=8,
    nperseg=1024,
    file_type="raw",
    title="Waterfall ? FFT evolution across runs",
)
bokeh_show(fig)
